# Lesson 2 - Vector Search - Code Along 2 - sqlitesearch

Make the vector search persistant using an SQLite database and optimize it to
use approximate nearest neighbour search (ANN) using sqlitesearch.
Here, I will index the data and save it to a db file.
I will load and use it in the next notebook.

In [1]:
# dependencies

from dotenv import load_dotenv
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

from ingest import load_faq_data

In [2]:
# load environment variables
load_dotenv()

True

In [3]:
# load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# get the course FAQ dataset
documents = load_faq_data()

# generate embeddings for the FAQ dataset
# embedd question and answer together so a query can match both
texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)
    
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

100%|██████████| 27/27 [00:05<00:00,  5.31it/s]


In [5]:
# index using ivf: k-means clustering for approximate candidate retrieval
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors.db",
)

# clear index
# remove all data from the db
# if this is not done, this code will return an error
# I assume I will have a good reason if I run this again, so include this step
# at least I want this to run smooth, and clearing will help by preventing error
vs_index.clear()

# fit index
vs_index.fit(vectors, documents)

In [6]:
# search a query
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)
results = vs_index.search(
    query_vector=query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [7]:
# check the results
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespace

In [8]:
# close connection
vs_index.close()